In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("tidyverse", "ggformula", "agricolae", "ggplot2", "dplyr")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)


── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: scales


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


Loading required package: ggiraph

Loading required package: ggridges


New to ggformula?  Try the tutorials: 
	learnr::run_tutorial("introduction", package = "ggform

[[1]]
 [1] "lubridate" "forcats"   "stringr"   "dplyr"     "purrr"     "readr"    
 [7] "tidyr"     "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"    
[13] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[2]]
 [1] "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate" "forcats"  
 [7] "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"   
[13] "ggplot2"   "tidyverse" "repr"      "stats"     "graphics"  "grDevices"
[19] "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[5]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"

# Visualize nucleotide diversity of PPR, NLR, and TAS genes

In [2]:
# load input files
prot_temp <- read.delim2("./lib/all_gene_nuc_div.txt", row.names = 1, header = TRUE, sep = "\t")
ppr_class <- read.delim2("./lib/ppr_class.txt", sep = "\t", header = TRUE)
nlr_class <- read.delim2("./lib/nlr_class.txt", sep = "\t", header = TRUE)


In [3]:
# process PPR data
ppr_temp <- prot_temp[match(ppr_class$ID, rownames(prot_temp)), ]
ppr_temp$Class <- ppr_class$Class[match(rownames(ppr_temp), ppr_class$ID)]
ppr_temp$ID <- rownames(ppr_temp)

# process NLR data
nlr_temp <- prot_temp[match(nlr_class$ID, rownames(prot_temp)), ]
nlr_temp$Class <- nlr_class$Class[match(rownames(nlr_temp), nlr_class$ID)]
nlr_temp$Class[is.na(nlr_temp$Class)] <- "non-siRNA-NLR"
nlr_temp$ID <- rownames(nlr_temp)

prot_temp$Class <- "all-protein"
prot_temp$ID <- rownames(prot_temp)

In [4]:
# generate random data
set.seed(123)  # for reproducibility

n_sample <- 500
n_iter <- 1000

ran_pi <- ran_Tajima.D <- ran_theta <- ran_dis <- matrix(0, nrow = n_sample, ncol = n_iter)

for (i in 1:n_iter) {
  ran_ids <- sample(1:nrow(prot_temp), n_sample, replace = FALSE)
  ran_pi[, i] <- prot_temp$Pi[ran_ids]
  ran_Tajima.D[, i] <- prot_temp$Tajima.D[ran_ids]
  ran_theta[, i] <- prot_temp$theta_Watterson[ran_ids]
  ran_dis[, i] <- prot_temp$Distance[ran_ids]
}

ran_pi <- apply(ran_pi, c(1, 2), as.numeric)
ran_Tajima.D <- apply(ran_Tajima.D, c(1, 2), as.numeric)
ran_theta <- apply(ran_theta, c(1, 2), as.numeric)
ran_dis <- apply(ran_dis, c(1, 2), as.numeric)

ran_temp <- suppressWarnings(
  data.frame(
    Pi = apply(ran_pi, 1, median, na.rm = TRUE),
    Tajima.D = apply(ran_Tajima.D, 1, median, na.rm = TRUE),
    theta_Watterson = apply(ran_theta, 1, median, na.rm = TRUE),
    Distance = apply(ran_dis, 1, median, na.rm = TRUE),
    Class = "random",
    ID = paste0("random_", 1:n_sample)
  )
)

In [5]:
# combine all data
pdata <- rbind(ppr_temp, nlr_temp)
pdata <- rbind(pdata, ran_temp)

level <- c("HS-siRNA-PPR", "non-HS-siRNA-PPR", "non-siRNA-PPR", 
           "siRNA-NLR", "non-siRNA-NLR", "random")
pdata$Class <- factor(pdata$Class, levels = level)
pdata$Pi <- as.numeric(pdata$Pi)


In [6]:
# Kruskal-Wallis test
test <- kruskal(pdata$Pi, pdata$Class, group = TRUE, p.adj = "bonferroni", alpha = 0.05)
test 


$statistics
     Chisq Df p.chisq
  192.1181  5       0

$parameters
            test  p.ajusted      name.t ntr alpha
  Kruskal-Wallis bonferroni pdata$Class   6  0.05

$means
                    pdata.Pi      rank          std   r         Min         Max
HS-siRNA-PPR     0.034892204 1003.5714 0.0231327277  14 0.000540541 0.081729323
non-HS-siRNA-PPR 0.023450501 1004.2500 0.0203713803   4 0.005738095 0.046090226
non-siRNA-NLR    0.023174580  863.1327 0.0289540372 162 0.000000000 0.154975369
non-siRNA-PPR    0.004873742  490.6849 0.0070892177 457 0.000000000 0.076959184
random           0.002688753  533.6310 0.0001619293 500 0.002251021 0.003214286
siRNA-NLR        0.014606139  639.5000 0.0162171131   4 0.001462585 0.035462185
                         Q25         Q50         Q75
HS-siRNA-PPR     0.014567669 0.036973684 0.046546053
non-HS-siRNA-PPR 0.006481516 0.020986842 0.037955827
non-siRNA-NLR    0.004523274 0.013116408 0.029564732
non-siRNA-PPR    0.001135531 0.002310924 0.00611111

In [7]:
t_comp <- test$means %>%
  rownames_to_column(var = "group") %>%
  rename(Pi = pdata.Pi) %>%
  as_tibble() %>%
  left_join(as_tibble(test$groups), by = c("rank" = "pdata$Pi"))

t_comp$group <- factor(t_comp$group, levels = level)
t_comp

group,Pi,rank,std,r,Min,Max,Q25,Q50,Q75,groups
<fct>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
HS-siRNA-PPR,0.034892204,1003.5714,0.0231327277,14,0.000540541,0.081729323,0.014567669,0.036973684,0.046546053,a
non-HS-siRNA-PPR,0.023450501,1004.2500,0.0203713803,4,0.005738095,0.046090226,0.006481516,0.020986842,0.037955827,a
non-siRNA-NLR,0.023174580,863.1327,0.0289540372,162,0.000000000,0.154975369,0.004523274,0.013116408,0.029564732,a
non-siRNA-PPR,0.004873742,490.6849,0.0070892177,457,0.000000000,0.076959184,0.001135531,0.002310924,0.006111111,b
random,0.002688753,533.6310,0.0001619293,500,0.002251021,0.003214286,0.002574535,0.002677295,0.002806608,b
siRNA-NLR,0.014606139,639.5000,0.0162171131,4,0.001462585,0.035462185,0.001900150,0.010749893,0.023455883,ab


In [8]:

p1 <- pdata %>% 
  ggplot(aes(x = Class, y = Pi, fill = Class)) +
  geom_boxplot(outlier.size = -1, width = 0.3) +
  geom_jitter(aes(group = Class), color = "black", size = 0.3,
              position = position_jitter(width = 0.2), alpha = 0.2) +
  scale_fill_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  ylim(c(0, 0.12)) +
  ylab("Nucleotide diversity, Pi") +
  xlab("") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.ticks = element_line(colour = "black")
  )

# save plot
ggsave(p1, file = "nuc_div_pi_boxplot.pdf", width = 4.8, height = 3)


Warning message:
“Removed 5 rows containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 7 rows containing missing values or values outside the scale range (`geom_point()`).”


In [9]:

p2 <- gf_histogram(~ Pi | Class, alpha = 0.2, data = pdata, bins = 30) %>%
  gf_freqpoly(~ Pi, data = pdata, color = ~ Class, size = 1) +
  facet_wrap(~ Class, scales = "free_y", nrow = 1) +
  scale_colour_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  scale_x_continuous(limits = c(0, 0.15), breaks = seq(0, 0.15, 0.03)) +
  xlab("Nucleotide diversity, Pi") +
  ylab("Loci Number") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
    legend.position = "none",
    strip.background = element_rect(colour = "white", fill = "white")
  )

# save plot
ggsave(p2, file = "nuc_div_pi_density.pdf", width = 10, height = 2)



Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”
Warning message:
“Removed 2 rows containing non-finite outside the scale range (`stat_bin()`).”
`stat_bin()` using `bins = 30`. Pick better value `binwidth`.
Warning message:
“Removed 2 rows containing non-finite outside the scale range (`stat_bin()`).”
Warning message:
“Removed 12 rows containing missing values or values outside the scale range (`geom_bar()`).”


# Visualize proportion of unique siRNAs

In [10]:
ppr_class <- read.delim2("./lib/ppr_class.txt", sep = "\t", header = TRUE)
uni_data <- read.delim2("./lib/unique_srna_accs.txt", sep = "\t", header = TRUE)

uni_data$TPM <- as.numeric(uni_data$TPM) / 10000
uni_data <- aggregate(uni_data$TPM, list(uni_data$Loci, uni_data$Accession), mean)
names(uni_data) <- c("Class", "Accession", "Ratio")
uni_data$Class <- factor(uni_data$Class, levels = c("PPR", "NLR", "TAS"))
uni_data$Accession <- factor(uni_data$Accession)


In [11]:
test <- kruskal(uni_data$Ratio, uni_data$Class, group = TRUE, p.adj = "bonferroni", alpha = 0.05)
test

$statistics
   Chisq Df      p.chisq t.value      MSD
  15.695  2 0.0003907276 2.60135 5.424432

$parameters
            test  p.ajusted         name.t ntr alpha
  Kruskal-Wallis bonferroni uni_data$Class   3  0.05

$means
    uni_data.Ratio   rank        std r       Min       Max        Q25       Q50
NLR      0.1101952  4.750 0.06755904 8 0.0233013 0.1924250 0.04331239 0.1273688
PPR      0.3604374 14.375 0.17822771 8 0.1852620 0.7095825 0.21590463 0.3274710
TAS      0.4443504 18.375 0.08484443 8 0.3231165 0.6128110 0.40551225 0.4347867
          Q75
NLR 0.1546485
PPR 0.4183716
TAS 0.4599171

$comparison
NULL

$groups
    uni_data$Ratio groups
TAS         18.375      a
PPR         14.375      a
NLR          4.750      b

attr(,"class")
[1] "group"

In [12]:

p <- ggplot(uni_data, aes(x = Class, y = Ratio)) +
  geom_boxplot(color = "black", outlier.shape = NA) +
  geom_jitter(aes(color = Accession), size = 2, position = position_jitter(width = 0.21)) +
  scale_colour_manual(values = alpha(c(
    "An-1" = "#0000FF", "Col-0" = "#FFA500", "Ct-1" = "#FF0000",
    "Cvi-1" = "#800000", "Eri-1" = "#000080", "Kyo-1" = "#008000",
    "Ler-0" = "#800080", "Sha" = "#696969"
  ), 0.8)) +
  labs(y = "Proportion of unique siRNAs (%)", x = NULL) +
  ylim(c(0, 1)) +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))

ggsave(p, file = "prop_unique_sRNA.pdf", width = 3.5, height = 2.3)


In [13]:
sessionInfo()

R version 4.5.1 (2025-06-13)
Platform: aarch64-apple-darwin23.6.0
Running under: macOS Sonoma 14.5

Matrix products: default
BLAS:   /opt/homebrew/Cellar/openblas/0.3.30/lib/libopenblasp-r0.3.30.dylib 
LAPACK: /opt/homebrew/Cellar/r/4.5.1/lib/R/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] en_GB.UTF-8/en_GB.UTF-8/en_GB.UTF-8/C/en_GB.UTF-8/en_GB.UTF-8

time zone: America/Chicago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] agricolae_1.3-7 ggformula_1.0.1 ggridges_0.5.7  ggiraph_0.9.3  
 [5] scales_1.4.0    lubridate_1.9.4 forcats_1.0.1   stringr_1.6.0  
 [9] dplyr_1.1.4     purrr_1.2.0     readr_2.1.6     tidyr_1.3.2    
[13] tibble_3.3.0    ggplot2_4.0.1   tidyverse_2.0.0 repr_1.1.7     

loaded via a namespace (and not attached):
 [1] generics_0.1.4          fontLiberation_0.1.0    lattice_0.22-7         
 [4] stringi_1.8.7           hms_1.1.4               digest_0